In [1]:
import pandas as pd

path = "../mockdata/"

df_waterings = pd.read_csv(f'{path}waterings.csv')
df_sensors = pd.read_csv(f'{path}sensor_datas.csv')
df_plants = pd.read_csv(f'{path}plants.csv')


df_waterings


,PlantMAC,PredictedFutureWaterTime,LastWaterTime,WaterLevel,PumpTimeInSeconds
0,34:2c:d8:10:0f:2f,2025-01-05T18:04:55.005251,2025-01-01T12:00:00,92.74,5
1,34:2c:d8:10:0f:2f,2025-01-08T10:15:21.865877,2025-01-05T18:04:55.005251,86.91,7
2,34:2c:d8:10:0f:2f,2025-01-10T12:00:52.926376,2025-01-08T10:15:21.865877,67.97,19
3,34:2c:d8:10:0f:2f,2025-01-13T22:13:28.266798,2025-01-10T12:00:52.926376,69.79,18
4,34:2c:d8:10:0f:2f,2025-01-16T03:28:05.343803,2025-01-13T22:13:28.266798,93.99,5
...,...,...,...,...,...
145,3a:d5:eb:27:3b:43,2025-01-24T05:22:54.931169,2025-01-22T03:24:45.579617,78.13,13
146,3a:d5:eb:27:3b:43,2025-01-28T03:34:35.710127,2025-01-24T05:22:54.931169,62.08,22
147,3a:d5:eb:27:3b:43,2025-02-01T12:31:47.917304,2025-01-28T03:34:35.710127,85.21,8
148,3a:d5:eb:27:3b:43,2025-02-04T17:22:39.468818,2025-02-01T12:31:47.917304,65.44,20


In [2]:
mac_map = {mac: i for i, mac in enumerate(df_plants['MAC'].unique())}

df_waterings = df_waterings.drop(columns=['PredictedFutureWaterTime'])
df_waterings = df_waterings.rename(columns={'LastWaterTime': 'Timestamp'})

df_sensors['Timestamp']   = pd.to_datetime(df_sensors['Timestamp'])
df_waterings['Timestamp'] = pd.to_datetime(df_waterings['Timestamp'], format='mixed')

df_sensors['mac_id']   = df_sensors['PlantMAC'].map(mac_map)
df_waterings['mac_id'] = df_waterings['PlantMAC'].map(mac_map)
df_plants['mac_id']    = df_plants['MAC'].map(mac_map)

df_plants_cleaned = df_plants.drop(columns=['MAC', 'Username', 'Name'])

sensors_sorted   = df_sensors.sort_values('Timestamp').reset_index(drop=True).drop(columns=['PlantMAC'])
waterings_sorted = df_waterings.sort_values('Timestamp').reset_index(drop=True).drop(columns=['PlantMAC'])

df_water_sensors = pd.merge_asof(
    sensors_sorted,
    waterings_sorted[['mac_id', 'Timestamp', 'PumpTimeInSeconds']].rename(columns={'Timestamp': 'LastWaterTimestamp'}),
    left_on='Timestamp',
    right_on='LastWaterTimestamp',
    by='mac_id',
    direction='backward'
)

df_water_sensors['seconds_since_watering'] = (
    df_water_sensors['Timestamp'] - df_water_sensors['LastWaterTimestamp']
).dt.total_seconds()

df_water_sensors = df_water_sensors.drop(columns=['LastWaterTimestamp', 'Timestamp'])

# Bring in plant optimal values
df_water_sensors = df_water_sensors.merge(df_plants_cleaned, on='mac_id', how='left')

df_water_sensors

,Temperature,AirHumidity,SoilHumidity,LightIntensity,mac_id,PumpTimeInSeconds,seconds_since_watering,OptimalTemperature,OptimalAirHumidity,OptimalSoilHumidity,OptimalLightIntensity
0,23.64,45.55,62.13,418.42,0,5,0.000000,23.1,40.8,58.3,411.6
1,27.49,55.65,74.27,568.89,12,6,0.000000,25.3,51.8,71.0,632.6
2,18.14,39.43,77.46,885.89,3,11,0.000000,19.2,42.4,69.9,758.4
3,16.37,51.70,84.73,462.77,9,16,0.000000,19.2,51.8,76.7,594.8
4,25.34,40.04,84.04,638.06,1,5,0.000000,24.2,46.6,74.1,648.0
...,...,...,...,...,...,...,...,...,...,...,...
1495,21.22,50.24,78.82,738.27,1,7,95844.928134,24.2,46.6,74.1,648.0
1496,26.87,53.15,63.96,786.51,12,21,79399.368184,25.3,51.8,71.0,632.6
1497,25.50,43.29,54.77,370.23,0,5,356400.000000,23.1,40.8,58.3,411.6
1498,19.63,50.18,79.60,649.45,9,17,108080.433356,19.2,51.8,76.7,594.8


In [3]:
for mac, group in df_water_sensors.groupby('mac_id'):
    pass

X = df_water_sensors.drop(columns=['mac_id', 'PumpTimeInSeconds'])
y = df_water_sensors['PumpTimeInSeconds']
X

,Temperature,AirHumidity,SoilHumidity,LightIntensity,seconds_since_watering,OptimalTemperature,OptimalAirHumidity,OptimalSoilHumidity,OptimalLightIntensity
0,23.64,45.55,62.13,418.42,0.000000,23.1,40.8,58.3,411.6
1,27.49,55.65,74.27,568.89,0.000000,25.3,51.8,71.0,632.6
2,18.14,39.43,77.46,885.89,0.000000,19.2,42.4,69.9,758.4
3,16.37,51.70,84.73,462.77,0.000000,19.2,51.8,76.7,594.8
4,25.34,40.04,84.04,638.06,0.000000,24.2,46.6,74.1,648.0
...,...,...,...,...,...,...,...,...,...
1495,21.22,50.24,78.82,738.27,95844.928134,24.2,46.6,74.1,648.0
1496,26.87,53.15,63.96,786.51,79399.368184,25.3,51.8,71.0,632.6
1497,25.50,43.29,54.77,370.23,356400.000000,23.1,40.8,58.3,411.6
1498,19.63,50.18,79.60,649.45,108080.433356,19.2,51.8,76.7,594.8
